# Ovarian Cancer — Platinum Treatment Response Prediction
**Dataset:** TCGA OV PanCancer Atlas 2018  
**Goal:** Predict platinum chemotherapy response (12m / 18m / 24m) from RNA-seq gene expression  
**Models:** Logistic Regression · Random Forest · SVM  

---
This notebook is a **clean, reproducible walkthrough** of the full pipeline.  
All logic lives in `src/` — this notebook calls those modules and displays results.


In [ ]:
# ── Environment setup ─────────────────────────────────────────────────────
import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO,
                    format='%(levelname)-8s | %(name)s | %(message)s')

# Project root (works whether run from notebooks/ or root)
ROOT = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()

import sys
sys.path.insert(0, str(ROOT))

print(f'Project root: {ROOT}')

In [ ]:
# ── Load configuration ─────────────────────────────────────────────────────
with open(ROOT / 'config' / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

# Override data path if needed
# cfg['data']['data_dir'] = '/path/to/your/data'

print(f"Project: {cfg['project']['name']}")
print(f"Thresholds: {cfg['clinical']['thresholds']} months")
print(f"Top genes: {cfg['features']['top_n_genes']}")
print(f"Random seed: {cfg['project']['seed']}")

---
## 1. Data Loading

In [ ]:
from src.data_loader import load_clinical, load_expression, load_treatment

data_dir = cfg['data']['data_dir']

clinical  = load_clinical(data_dir, cfg['data']['clinical_file'])
treatment = load_treatment(data_dir, cfg['data']['treatment_file'])
expr_raw  = load_expression(data_dir, cfg['data']['expression_file'])

print(f'Clinical:   {clinical.shape}')
print(f'Treatment:  {treatment.shape}')
print(f'Expression: {expr_raw.shape}')
clinical.head(3)

---
## 2. Preprocessing

In [ ]:
from src.preprocessing import (
    extract_survival,
    identify_platinum_patients,
    build_labelled_survival,
    clean_expression,
    align_expression_to_labels,
)

# 2a. Extract and clean survival
surv = extract_survival(clinical, cfg['clinical']['time_col'], cfg['clinical']['status_col'])

# 2b. Identify platinum-treated patients
platinum_ids = identify_platinum_patients(treatment, cfg['clinical']['platinum_drugs'])

# 2c. Create binary response labels at each threshold
labelled_dict = build_labelled_survival(
    surv, platinum_ids,
    cfg['clinical']['time_col'],
    cfg['clinical']['thresholds']
)

# 2d. Clean expression matrix
expr = clean_expression(expr_raw)

print('\nLabel counts per threshold:')
for thr, df in labelled_dict.items():
    print(f'  {thr}m → {df["response"].value_counts().to_dict()}')

In [ ]:
# Visualise class balance
from src.visualization import plot_class_distribution

label_series = {thr: df['response'] for thr, df in labelled_dict.items()}
_ = plot_class_distribution(label_series, save_path=ROOT / 'outputs/figures/fig1_class_distribution.png')

---
## 3. Feature Engineering

**Key decision:** Variance-based gene selection is computed **on the training split only** to prevent information leakage — a common mistake in genomic ML papers.

In [ ]:
from sklearn.model_selection import train_test_split
from src.features import select_high_variance_genes, scale_features

feature_sets = {}

for thr, labelled in labelled_dict.items():
    X, y = align_expression_to_labels(expr, labelled)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=cfg['model']['test_size'],
        random_state=cfg['project']['seed'],
        stratify=y
    )

    X_train_red, X_test_red = select_high_variance_genes(
        X_train, X_test, top_n=cfg['features']['top_n_genes']
    )

    X_train_sc, X_test_sc, scaler = scale_features(X_train_red, X_test_red)

    feature_sets[thr] = dict(
        X_train=X_train_red, X_test=X_test_red,
        X_train_scaled=X_train_sc, X_test_scaled=X_test_sc,
        y_train=y_train, y_test=y_test, scaler=scaler
    )
    print(f'{thr}m → train: {X_train_sc.shape} | test: {X_test_sc.shape}')

---
## 4. Model Training & Cross-Validation

In [ ]:
from src.models import get_all_models, train_model, cross_validate_model

trained_models = {}
cv_results = {}

for thr, fs in feature_sets.items():
    print(f'\n── {thr}-month endpoint ──')
    models = get_all_models(cfg)
    thr_trained = {}
    cv_results[thr] = {}

    for name, model in models.items():
        fitted = train_model(model, fs['X_train_scaled'], fs['y_train'])
        thr_trained[name] = fitted

        cv = cross_validate_model(
            fitted, fs['X_train_scaled'], fs['y_train'],
            n_splits=cfg['model']['cv_folds'],
            seed=cfg['project']['seed']
        )
        cv_results[thr][name] = cv
        print(f'  {name}: CV ROC-AUC = {cv["mean"]:.3f} ± {cv["std"]:.3f}')

    trained_models[thr] = thr_trained

---
## 5. Evaluation

In [ ]:
from src.evaluation import evaluate_all_models, build_results_table

all_results = []
for thr, fs in feature_sets.items():
    results = evaluate_all_models(trained_models[thr], fs['X_test_scaled'], fs['y_test'], threshold=thr)
    all_results.extend(results)

results_table = build_results_table(all_results)
results_table

---
## 6. Visualisations
### 6a. ROC Curves

In [ ]:
from src.visualization import plot_roc_curves, plot_confusion_matrices, plot_cv_auc_heatmap

for thr in feature_sets:
    res = [r for r in all_results if r['threshold'] == thr]
    probs = {r['model_name'].split(' (')[0]: r['y_prob'] for r in res}
    y_test = feature_sets[thr]['y_test']

    plot_roc_curves(
        y_test, probs,
        title=f'ROC Curves — {thr}-month endpoint',
        save_path=ROOT / f'outputs/figures/fig2_roc_{thr}m.png'
    )
    plt.show()

### 6b. Confusion Matrices

In [ ]:
for thr in feature_sets:
    res = [r for r in all_results if r['threshold'] == thr]
    preds = {r['model_name'].split(' (')[0]: r['y_pred'] for r in res}
    y_test = feature_sets[thr]['y_test']

    plot_confusion_matrices(
        y_test, preds,
        title=f'Confusion Matrices — {thr}-month endpoint',
        save_path=ROOT / f'outputs/figures/fig3_confusion_{thr}m.png'
    )
    plt.show()

### 6c. CV AUC Heatmap

In [ ]:
plot_cv_auc_heatmap(cv_results, save_path=ROOT / 'outputs/figures/fig4_cv_auc_heatmap.png')
plt.show()

### 6d. Feature Importance (Random Forest)

In [ ]:
from src.visualization import plot_feature_importance

for thr, models in trained_models.items():
    rf = models['Random Forest']
    feat_names = np.array(feature_sets[thr]['X_train'].columns)
    plot_feature_importance(
        feat_names, rf.feature_importances_,
        title=f'Top Predictive Genes — RF {thr}m',
        save_path=ROOT / f'outputs/figures/fig5_feature_importance_{thr}m.png'
    )
    plt.show()

### 6e. Calibration Curves

In [ ]:
from src.visualization import plot_calibration_curves

calib_data = {}
for thr, models in trained_models.items():
    rf = models['Random Forest']
    y_test = feature_sets[thr]['y_test']
    y_prob = rf.predict_proba(feature_sets[thr]['X_test_scaled'])[:, 1]
    calib_data[f'RF {thr}m'] = (y_test, y_prob)

plot_calibration_curves(calib_data, save_path=ROOT / 'outputs/figures/fig6_calibration.png')
plt.show()

---
## 7. SHAP Explainability (Best Model)

In [ ]:
import shap

# Explain the 24m Random Forest (typically best AUC)
best_thr = 24
rf = trained_models[best_thr]['Random Forest']
X_explain = feature_sets[best_thr]['X_train']

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_explain)

# SHAP values for positive class (index 1)
shap_pos = shap_values[:, :, 1] if shap_values.ndim == 3 else shap_values[1]

print(f'SHAP values shape: {shap_pos.shape}')

In [ ]:
# Global importance — bar plot
shap.summary_plot(shap_pos, X_explain, plot_type='bar', max_display=20, show=True)

In [ ]:
# Beeswarm — direction + magnitude
shap.summary_plot(shap_pos, X_explain, max_display=20, show=True)

---
## 8. Survival Analysis (Kaplan–Meier)

In [ ]:
from src.visualization import plot_kaplan_meier

# Align survival data to 24m test patients
best_thr = 24
fs = feature_sets[best_thr]
best_model = trained_models[best_thr]['Logistic Regression']

y_prob_km = best_model.predict_proba(fs['X_test_scaled'])[:, 1]
y_pred_km = (y_prob_km >= 0.5).astype(int)

# Build survival dataframe for test patients
test_patient_ids = fs['X_test'].index
surv_test = surv[surv['PATIENT_ID'].isin(test_patient_ids)].set_index('PATIENT_ID')

surv_df = pd.DataFrame({
    'pred_class': y_pred_km,
    'pred_prob': y_prob_km,
    'time': surv_test.loc[test_patient_ids, cfg['clinical']['time_col']].values,
    'event': surv_test.loc[test_patient_ids, 'event'].values
})

plot_kaplan_meier(
    surv_df,
    title=f'Kaplan–Meier: Predicted Risk Groups ({best_thr}m endpoint)',
    save_path=ROOT / 'outputs/figures/fig7_kaplan_meier_24m.png'
)
plt.show()

---
## 9. Final Results Summary

In [ ]:
print('=' * 70)
print('FINAL MODEL PERFORMANCE SUMMARY')
print('=' * 70)
print(results_table.to_string(index=False))

best_row = results_table.nlargest(1, 'roc_auc').iloc[0]
print(f'\n✓ Best model: {best_row["model"]} at {best_row["threshold_m"]}m threshold')
print(f'  ROC-AUC: {best_row["roc_auc"]:.4f}')

results_table.to_csv(ROOT / 'outputs/results/model_performance_summary.csv', index=False)
print('\nResults saved to outputs/results/model_performance_summary.csv')